In [79]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import IntegerType, DoubleType

In [80]:
spark = SparkSession.builder \
    .appName("Process Sales Pipeline") \
    .getOrCreate()

In [81]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

In [82]:
spark.conf.set("spark.sql.ansi.enabled", False)

In [83]:
base_path = Path("/app")
bronze_path = base_path / "bronze" / "sales"
silver_path = base_path / "silver" / "sales"

In [89]:
df_bronze = spark.read.parquet(str(bronze_path))

In [90]:
print(df_bronze.columns)


['CustomerId', 'PurchaseDate', 'Product', 'Price']


In [104]:
df_silver = df_bronze \
    .withColumnRenamed("CustomerId", "client_id") \
    .withColumnRenamed("PurchaseDate", "purchase_date") \
    .withColumnRenamed("Product", "product_name") \
    .withColumnRenamed("Price", "price") \
    .withColumn("client_id", col("client_id").cast(IntegerType())) \
    .withColumn(
        "purchase_date",
        coalesce(
            to_date(regexp_replace(col("purchase_date"), "/", "-"), "yyyy-MM-dd"),
            to_date(col("purchase_date"), "yyyy-M-d"),
            to_date(col("purchase_date"), "yyyy-MMM-dd")
        )
    ) \
    .withColumn("price", regexp_replace(col("price"), "[^0-9.]", "").cast(DoubleType())) \
    .filter(
        col("client_id").isNotNull() &
        col("purchase_date").isNotNull() &
        col("price").isNotNull()
    )



In [105]:
df_bronze.select("PurchaseDate").distinct().show(50, False)



+------------+
|PurchaseDate|
+------------+
|2022-09-29  |
|2022-09-23  |
|2022-09-27  |
|2022-Aug-30 |
|2022/09/30  |
|2022-09-24  |
|2022-09-18  |
|2022-09-22  |
|2022-09-3   |
|2022-09-1   |
|2022-09-13  |
|2022-09-9   |
|2022-09-6   |
|2022-09-16  |
|2022-09-12  |
|2022-09-11  |
|2022-09-10  |
|2022-09-8   |
|2022-09-5   |
|2022-09-26  |
|2022-09-28  |
|2022-09-17  |
|2022-09-14  |
|2022/09/14  |
|2022-09-19  |
|2022-09-25  |
|2022-09-20  |
|2022-09-21  |
|2022-09-7   |
|2022-09-2   |
|2022-09-15  |
|2022-09-4   |
+------------+



In [107]:
df_silver.write \
    .mode("overwrite") \
    .partitionBy("purchase_date") \
    .parquet(str(silver_path))

26/02/15 19:35:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/15 19:35:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/15 19:35:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/02/15 19:35:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/02/15 19:35:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/02/15 19:35:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/02/15 19:35:28 WARN MemoryManager: Total allocation exceeds 95.

In [108]:
print("Bronze rows:", df_bronze.count())
print("Silver rows:", df_silver.count())


Bronze rows: 71085
Silver rows: 71085


In [109]:
df_silver.select("purchase_date").distinct().show(20, False)

+-------------+
|purchase_date|
+-------------+
|2022-09-29   |
|2022-09-23   |
|2022-09-27   |
|2022-09-30   |
|2022-08-30   |
|2022-09-24   |
|2022-09-18   |
|2022-09-03   |
|2022-09-22   |
|2022-09-01   |
|2022-09-13   |
|2022-09-06   |
|2022-09-09   |
|2022-09-12   |
|2022-09-16   |
|2022-09-10   |
|2022-09-11   |
|2022-09-05   |
|2022-09-08   |
|2022-09-28   |
+-------------+
only showing top 20 rows


In [110]:
print("Sales pipeline completed successfully!")

Sales pipeline completed successfully!


In [111]:
spark.stop()